# Module 4. `module3_sft_dataset.jsonl`로 SFTTrainer 기반 SFT 실습

이 노트북의 목표는 다음 4가지입니다.

1. `module3_sft_dataset.jsonl`을 Colab에서 불러옵니다.  
2. TRL의 `SFTTrainer`로 실제 SFT를 수행합니다.  
3. 튜닝 전후 출력을 비교합니다.  
4. 학습 결과와 관찰 메모를 저장합니다.

## 이번 모듈의 핵심
- 데이터 형식: `prompt-completion`
- 학습 방식: `SFTTrainer`
- 실습 전략: Colab 단일 GPU 기준 LoRA/PEFT 활용
- 비교 포인트: baseline 대비 말투, 형식, 간단한 task 수행 변화

## 준비물
- Module 3에서 만든 `module3_sft_dataset.jsonl`
- GPU가 활성화된 Colab 런타임 권장

## Step 1. 런타임 확인

먼저 현재 Colab 환경에서 Python, PyTorch, GPU 상태를 확인합니다.

### 확인 포인트
- GPU가 보이면 실습 속도가 훨씬 안정적입니다.
- GPU가 없더라도 코드는 실행 가능하지만 시간이 오래 걸릴 수 있습니다.

In [2]:
import torch
import platform
import sys

print("Python version:", sys.version)
print("Platform:", platform.platform())
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not detected. Colab 메뉴에서 Runtime > Change runtime type > GPU 설정을 권장합니다.")

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## Step 2. 필수 라이브러리 설치

이번 실습에서 사용할 핵심 라이브러리를 설치합니다.

- `transformers`: 모델/토크나이저 로드
- `datasets`: JSONL 데이터셋 로드
- `trl`: `SFTTrainer`
- `peft`: LoRA/PEFT 설정
- `accelerate`: 학습 지원
- `sentencepiece`: tokenizer 지원

In [3]:
!pip -q install -U transformers datasets trl peft accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 134.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 20.2 MB/s eta 0:00:00


## Step 3. 라이브러리 import

학습과 저장, 비교를 위한 기본 라이브러리를 불러옵니다.

In [4]:
import os
import json
from typing import List, Dict, Any

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

In [1]:
import os

# GitHub 링크에서 파일 다운로드
url = 'https://github.com/JSJeong-me/TRL/raw/main/01%20Lecture/module3_sft_dataset_example.jsonl'
output_filename = 'module3_sft_dataset.jsonl'

# wget 명령어를 사용하여 파일 다운로드
# 이미 파일이 존재하면 덮어쓰거나 메시지를 출력하도록 -N 옵션 사용
!wget -N "{url}" -O "{output_filename}"

# 다운로드된 파일의 존재 여부 확인
if os.path.exists(output_filename):
    print(f"'{output_filename}' 파일이 성공적으로 다운로드되었습니다.")
    # DATA_PATH 변수를 업데이트하여 다운로드된 파일을 사용하도록 설정
    DATA_PATH = output_filename
else:
    raise FileNotFoundError(f"'{output_filename}' 파일을 다운로드하지 못했습니다.")

print("Using DATA_PATH:", DATA_PATH)

for details.

--2026-04-08 04:33:13--  https://github.com/JSJeong-me/TRL/raw/main/01%20Lecture/module3_sft_dataset_example.jsonl
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/JSJeong-me/TRL/main/01%20Lecture/module3_sft_dataset_example.jsonl [following]
--2026-04-08 04:33:13--  https://raw.githubusercontent.com/JSJeong-me/TRL/main/01%20Lecture/module3_sft_dataset_example.jsonl
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2317 (2.3K) [text/plain]
Saving to: ‘module3_sft_dataset.jsonl’

module3_sft_dataset 100%[===================>]   2.26K  --.-KB/s    in 0s      

2026-04-08 04:33:13 (3

## Step 4. 실험 설정

이번 실습에서는 **같은 조건으로 전후 비교**하는 것이 중요합니다.

### 기본 설정
- 기본 모델: `HuggingFaceTB/SmolLM2-360M-Instruct`
- 입력 데이터: `module3_sft_dataset.jsonl`
- 출력 폴더: `/content/module4_sft_output`

GPU 여유가 있으면 모델을 1.7B로 바꿔도 됩니다.

In [5]:
set_seed(42)

MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"
# MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

DEFAULT_DATA_PATH = "/content/module3_sft_dataset.jsonl"
OUTPUT_DIR = "/content/module4_sft_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("MODEL_NAME:", MODEL_NAME)
print("DEFAULT_DATA_PATH:", DEFAULT_DATA_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

MODEL_NAME: HuggingFaceTB/SmolLM2-360M-Instruct
DEFAULT_DATA_PATH: /content/module3_sft_dataset.jsonl
OUTPUT_DIR: /content/module4_sft_output


## Step 5. Module 3 데이터셋 준비

기본 경로에 `module3_sft_dataset.jsonl`이 있으면 그대로 사용합니다.  
파일이 없다면 직접 업로드할 수 있습니다.

### 기대 형식
각 줄은 아래와 같은 JSON 객체입니다.

```json
{"prompt": "27 * 14 의 결과만 답하세요.", "completion": "378"}
```

In [7]:
DATA_PATH = DEFAULT_DATA_PATH

if not os.path.exists(DATA_PATH):
    print(f"File not found at {DATA_PATH}")
    print("업로드 창을 열어 module3_sft_dataset.jsonl 파일을 직접 업로드하세요.")
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) == 0:
        raise FileNotFoundError("업로드된 파일이 없습니다.")
    uploaded_name = list(uploaded.keys())[0]
    DATA_PATH = f"/content/{uploaded_name}"

print("Using DATA_PATH:", DATA_PATH)

Using DATA_PATH: /content/module3_sft_dataset.jsonl


In [8]:
DATA_PATH = DEFAULT_DATA_PATH

if not os.path.exists(DATA_PATH):
    print(f"File not found at {DATA_PATH}")
    print("업로드 창을 열어 module3_sft_dataset.jsonl 파일을 직접 업로드하세요.")
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) == 0:
        raise FileNotFoundError("업로드된 파일이 없습니다.")
    uploaded_name = list(uploaded.keys())[0]
    DATA_PATH = f"/content/{uploaded_name}"

print("Using DATA_PATH:", DATA_PATH)

Using DATA_PATH: /content/module3_sft_dataset.jsonl


## Step 6. 데이터셋 로드

로컬 JSONL 파일을 Hugging Face `datasets`로 로드합니다.

In [9]:
dataset = load_dataset("json", data_files=DATA_PATH, split="train")
print(dataset)
print("First row:")
print(dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['prompt', 'completion', 'task_id', 'category'],
    num_rows: 8
})
First row:
{'prompt': '고객 문의에 정중한 한국어로 3문장 이내로 답하세요: 배송이 늦어지고 있습니다.', 'completion': '안녕하세요. 배송 지연으로 불편을 드려 죄송합니다. 현재 배송 상태를 확인 중이며, 확인되는 대로 빠르게 안내드리겠습니다.', 'task_id': 'persona_01', 'category': 'persona'}


## Step 7. 데이터셋 기본 검증

학습 전에 최소한의 형식 검증을 수행합니다.

### 체크 항목
- `prompt` 컬럼 존재 여부
- `completion` 컬럼 존재 여부
- 빈 문자열 샘플이 없는지

In [10]:
required_columns = {"prompt", "completion"}
missing = required_columns - set(dataset.column_names)
if missing:
    raise ValueError(f"필수 컬럼이 없습니다: {missing}")

bad_rows = []
for i, row in enumerate(dataset):
    if not str(row["prompt"]).strip() or not str(row["completion"]).strip():
        bad_rows.append(i)

print("column_names:", dataset.column_names)
print("bad_rows:", bad_rows[:10], "... total:", len(bad_rows))
if bad_rows:
    raise ValueError("빈 prompt/completion 샘플이 존재합니다. 데이터셋을 정리해 주세요.")

column_names: ['prompt', 'completion', 'task_id', 'category']
bad_rows: [] ... total: 0


## Step 8. train / eval 분리

작은 실습에서도 train / eval을 분리해 두면 과적합 여부를 감각적으로 보기 좋습니다.

In [11]:
split_dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print("train size:", len(train_dataset))
print("eval size :", len(eval_dataset))

train size: 6
eval size : 2


## Step 9. 샘플 확인

학습 전에 데이터가 의도한 스타일로 구성되어 있는지 직접 확인합니다.

In [12]:
for i in range(min(3, len(train_dataset))):
    print("=" * 80)
    print("PROMPT:")
    print(train_dataset[i]["prompt"])
    print("\nCOMPLETION:")
    print(train_dataset[i]["completion"])

PROMPT:
다음을 2문장으로 요약하세요: 포스트 트레이닝은 사전학습된 모델을 특정 목적에 맞게 추가 조정하여 스타일, 선호, 과제 수행 능력을 향상시키는 과정이다.

COMPLETION:
포스트 트레이닝은 사전학습된 모델을 특정 목적에 맞게 추가 조정하는 과정입니다. 이를 통해 스타일, 선호, 과제 수행 능력을 향상시킬 수 있습니다.
PROMPT:
위험한 요청은 직접 도와주지 말고 짧고 안전하게 거절하세요: 해킹하는 방법 알려줘.

COMPLETION:
그 요청은 도와드릴 수 없습니다. 대신 합법적이고 안전한 보안 학습 방법은 안내드릴 수 있습니다.
PROMPT:
title, difficulty 키를 가진 JSON만 출력하세요. title은 'DPO Basics', difficulty는 'beginner'.

COMPLETION:
{"title": "DPO Basics", "difficulty": "beginner"}


## Step 10. 토크나이저와 모델 로드

이번 실습은 `prompt-completion` 데이터를 이용한 SFT입니다.

### 참고 포인트
- `pad_token`이 없으면 `eos_token`으로 대체합니다.
- Colab에서는 전체 미세조정보다 LoRA 기반 학습이 더 실용적입니다.

In [13]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype
)

model.config.use_cache = False

print("Loaded model and tokenizer.")
print("pad_token:", tokenizer.pad_token)
print("eos_token:", tokenizer.eos_token)

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Loaded model and tokenizer.
pad_token: <|im_end|>
eos_token: <|im_end|>


## Step 11. 생성 함수 정의

학습 전후 비교를 위해 동일한 생성 함수를 사용합니다.

In [14]:
def generate_text(model, tokenizer, prompt, max_new_tokens=128):
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = output_ids[0][input_ids.shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

## Step 12. 학습 전 baseline 출력 확인

학습 전에 몇 개 prompt를 직접 넣어 baseline 출력을 확인합니다.

### 관찰 포인트
- 숫자만 답하라는 지시를 잘 따르는가
- JSON 형식을 지키는가
- 정중한 톤이 유지되는가

In [16]:
def generate_text(model, tokenizer, prompt, max_new_tokens=128):
    messages = [{"role": "user", "content": prompt}]
    encoded_input = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )
    input_ids = encoded_input["input_ids"].to(model.device)

    attention_mask = None
    if "attention_mask" in encoded_input:
        attention_mask = encoded_input["attention_mask"].to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            attention_mask=attention_mask, # Pass attention_mask if available
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = output_ids[0][input_ids.shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

test_prompts = [
    "27 * 14 의 결과만 답하세요.",
    "name, role 키를 가진 JSON만 출력하세요. name은 Alice, role은 engineer.",
    "고객 문의에 정중한 한국어로 3문장 이내로 답하세요: 배송이 늦어지고 있습니다."
]

before_outputs = []

print("=== BEFORE SFT ===")
for p in test_prompts:
    out = generate_text(model, tokenizer, p)
    before_outputs.append({"prompt": p, "output": out})
    print("=" * 80)
    print("PROMPT:", p)
    print("OUTPUT:", out)

=== BEFORE SFT ===
PROMPT: 27 * 14 의 결과만 답하세요.
OUTPUT: 27 * 14 = 418
PROMPT: name, role 키를 가진 JSON만 출력하세요. name은 Alice, role은 engineer.
OUTPUT: 자물속에서 일정한 말고, 도서가 개발의 도출에 대한 시모를 가질 수 있습니다.
PROMPT: 고객 문의에 정중한 한국어로 3문장 이내로 답하세요: 배송이 늦어지고 있습니다.
OUTPUT: 배송이 늦어지고 있습니다.


## Step 13. LoRA 설정

Colab 환경에서는 LoRA/PEFT를 이용하면 훨씬 가볍게 SFT를 진행할 수 있습니다.

### 기본 전략
- 전체 모델을 전부 업데이트하지 않고 adapter만 학습
- 교육용 실습에서는 반복성과 비용 측면에서 유리

In [17]:
peft_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear"
)

print(peft_config)

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.18.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules='all-linear', exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, arrow_config=None, ensure_weight_tying=False)


## Step 14. SFTConfig 설정

이번 실습은 `prompt-completion` 데이터셋을 사용하므로  
completion 부분에만 loss가 계산되도록 `completion_only_loss=True`를 명시합니다.

### 주요 설정
- `learning_rate=1e-4`: LoRA 실습용
- `gradient_accumulation_steps`: 작은 GPU에서 effective batch 확보
- `eval_strategy="epoch"`: epoch 단위 평가
- `save_strategy="epoch"`: epoch 종료 시 저장
- `packing=False`: 교육용으로 단순하게 유지

In [18]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=1e-4,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    max_length=512,
    packing=False,
    completion_only_loss=True,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
)

print(training_args)

SFTConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
activation_offloading=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
assistant_only_loss=False,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
chat_template_path=None,
completion_only_loss=True,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
dataset_kwargs=None,
dataset_num_proc=None,
dataset_text_field=text,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eos_tok

## Step 15. SFTTrainer 생성

이제 `train_dataset`, `eval_dataset`, `tokenizer`, `peft_config`를 이용해 trainer를 만듭니다.

In [19]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

print(trainer)

Adding EOS to train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

## Step 16. 학습 실행

이 셀에서 실제 SFT를 수행합니다.

### 실습 팁
- 데이터셋이 아주 작으면 loss가 빠르게 내려갈 수 있습니다.
- 학습 시간이 길다면 `num_train_epochs`를 1 또는 2로 줄여도 됩니다.

In [20]:
train_result = trainer.train()
print(train_result)

Epoch,Training Loss,Validation Loss
1,No log,7.686631
2,No log,7.679188
3,No log,7.675653


TrainOutput(global_step=3, training_loss=1.0589840412139893, metrics={'train_runtime': 17.6471, 'train_samples_per_second': 1.02, 'train_steps_per_second': 0.17, 'total_flos': 9098188780800.0, 'train_loss': 1.0589840412139893})


## Step 17. 모델 저장

학습이 끝나면 adapter와 tokenizer를 출력 폴더에 저장합니다.

In [21]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved to:", OUTPUT_DIR)

Saved to: /content/module4_sft_output


## Step 18. 학습 후 출력 비교

같은 prompt를 다시 넣어 전후 차이를 비교합니다.

In [22]:
trained_model = trainer.model

after_outputs = []

print("=== AFTER SFT ===")
for p in test_prompts:
    out = generate_text(trained_model, tokenizer, p)
    after_outputs.append({"prompt": p, "output": out})
    print("=" * 80)
    print("PROMPT:", p)
    print("OUTPUT:", out)

=== AFTER SFT ===
PROMPT: 27 * 14 의 결과만 답하세요.
OUTPUT: 27 * 14 = 398
PROMPT: name, role 키를 가진 JSON만 출력하세요. name은 Alice, role은 engineer.
OUTPUT: 명령 수정하신 것을 수정합니다. 자료를 적용해서 출력하세요. 이제 자료를 출력하고 있습니다.
PROMPT: 고객 문의에 정중한 한국어로 3문장 이내로 답하세요: 배송이 늦어지고 있습니다.
OUTPUT: 영상을 배송합니다.


## Step 19. 전후 비교표 저장

학습 전후 출력을 한 파일에 저장해 두면 이후 평가 모듈에서 재사용하기 쉽습니다.

In [23]:
comparison_rows = []
for before, after in zip(before_outputs, after_outputs):
    comparison_rows.append({
        "prompt": before["prompt"],
        "before": before["output"],
        "after": after["output"]
    })

comparison_path = os.path.join(OUTPUT_DIR, "module4_before_after_comparison.json")
with open(comparison_path, "w", encoding="utf-8") as f:
    json.dump(comparison_rows, f, ensure_ascii=False, indent=2)

print("Saved:", comparison_path)
comparison_rows

Saved: /content/module4_sft_output/module4_before_after_comparison.json


[{'prompt': '27 * 14 의 결과만 답하세요.',
  'before': '27 * 14 = 418',
  'after': '27 * 14 = 398'},
 {'prompt': 'name, role 키를 가진 JSON만 출력하세요. name은 Alice, role은 engineer.',
  'before': '자물속에서 일정한 말고, 도서가 개발의 도출에 대한 시모를 가질 수 있습니다.',
  'after': '명령 수정하신 것을 수정합니다. 자료를 적용해서 출력하세요. 이제 자료를 출력하고 있습니다.'},
 {'prompt': '고객 문의에 정중한 한국어로 3문장 이내로 답하세요: 배송이 늦어지고 있습니다.',
  'before': '배송이 늦어지고 있습니다.',
  'after': '영상을 배송합니다.'}]

## Step 20. 간단한 관찰 메모 템플릿 저장

아래 템플릿을 채워서 이번 SFT 실습 관찰 메모를 남깁니다.

In [24]:
notes = '''# Module 4 SFT Observation

## What changed after SFT?
-

## Which category improved the most?
-

## Which category still needs more work?
-

## Is SFT enough for this task?
-

## Candidate next step
- DPO / PPO / GRPO
- reason:
'''

notes_path = os.path.join(OUTPUT_DIR, "module4_sft_observation.md")
with open(notes_path, "w", encoding="utf-8") as f:
    f.write(notes)

print("Saved:", notes_path)
print(notes)

Saved: /content/module4_sft_output/module4_sft_observation.md
# Module 4 SFT Observation

## What changed after SFT?
- 

## Which category improved the most?
- 

## Which category still needs more work?
- 

## Is SFT enough for this task?
- 

## Candidate next step
- DPO / PPO / GRPO
- reason:



## Step 21. 결과 파일 확인

이번 모듈이 끝나면 아래 파일들이 생성되어야 합니다.

- 학습된 adapter / checkpoint
- `module4_before_after_comparison.json`
- `module4_sft_observation.md`

In [25]:
print("Saved files:")
for root, dirs, files in os.walk(OUTPUT_DIR):
    for name in files:
        print("-", os.path.join(root, name))

Saved files:
- /content/module4_sft_output/chat_template.jinja
- /content/module4_sft_output/module4_before_after_comparison.json
- /content/module4_sft_output/adapter_config.json
- /content/module4_sft_output/README.md
- /content/module4_sft_output/module4_sft_observation.md
- /content/module4_sft_output/tokenizer_config.json
- /content/module4_sft_output/tokenizer.json
- /content/module4_sft_output/adapter_model.safetensors
- /content/module4_sft_output/training_args.bin
- /content/module4_sft_output/checkpoint-3/chat_template.jinja
- /content/module4_sft_output/checkpoint-3/optimizer.pt
- /content/module4_sft_output/checkpoint-3/rng_state.pth
- /content/module4_sft_output/checkpoint-3/adapter_config.json
- /content/module4_sft_output/checkpoint-3/README.md
- /content/module4_sft_output/checkpoint-3/scheduler.pt
- /content/module4_sft_output/checkpoint-3/trainer_state.json
- /content/module4_sft_output/checkpoint-3/scaler.pt
- /content/module4_sft_output/checkpoint-3/tokenizer_config

## Step 22. 선택 사항: 결과 다운로드

Colab 세션이 종료되기 전에 주요 결과 파일을 내려받을 수 있습니다.

In [26]:
from google.colab import files

files.download(os.path.join(OUTPUT_DIR, "module4_before_after_comparison.json"))
files.download(os.path.join(OUTPUT_DIR, "module4_sft_observation.md"))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Module 4 정리

이번 모듈에서는 Module 3에서 만든 `prompt-completion` 데이터셋을 사용해 실제 SFT를 수행했습니다.

## 이번 모듈에서 확인한 것
- `SFTTrainer`는 `prompt-completion` 형식을 직접 사용할 수 있습니다.
- completion-only loss 설정은 답변 생성 학습에 집중하게 해 줍니다.
- Colab에서는 LoRA 기반 SFT가 실용적입니다.
- SFT는 스타일, 형식, 기본 과제 수행을 먼저 안정화하는 데 적합합니다.

## 다음 단계로 연결
다음 Module에서는 아래 두 가지 방향이 자연스럽습니다.

1. **평가 모듈**
   - baseline vs SFT 비교
   - 정성/정량 루브릭 작성

2. **DPO 모듈**
   - `module3_dpo_dataset.jsonl` 사용
   - 선호쌍 기반 정렬 실습

즉, Module 4는 “정답 예시로 먼저 고치는 단계”라고 이해하면 됩니다.